<img src=https://www.factset.com/hubfs/Assets/images/factset-logo.svg width="300" align="left">


# FactSet ESG API Example - Danish Companies

Extract ESG data (Truvalue Scores, Spotlights, Articles) for major Danish listed companies.

| Company | ISIN |
| --- | --- |
| Novo Nordisk B | DK0062498333 |
| DSV | DK0060079531 |
| Danske Bank | DK0010274414 |
| Vestas Wind Systems | DK0061539921 |
| Ørsted | DK0060094928 |
| Carlsberg B | DK0010181759 |
| A.P. Møller - Mærsk A | DK0010244425 |
| A.P. Møller - Mærsk B | DK0010244508 |
| Genmab | DK0010272202 |
| Coloplast | DK0060448595 |
| Tryg | DK0060636678 |
| Pandora | DK0060252690 |

## 1. Setup - Import packages and load credentials

In [ ]:
import requests
import json
import time
import pandas as pd
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
from pandas import json_normalize

import os
from dotenv import load_dotenv
load_dotenv()


In [ ]:
USERNAME = os.getenv("USERNAME")
APIKEY = os.getenv("APIKEY")
ESG_SCORES_URL = 'https://api.factset.com/content/factset-esg/v3/truvalue/scores'
ESG_SPOTLIGHTS_URL = 'https://api.factset.com/content/factset-esg/v3/truvalue/spotlights'
ESG_ARTICLES_URL = 'https://api.factset.com/content/factset-esg/v3/truvalue/articles'

START_DATE = "2023-01-01"
END_DATE = "2025-12-31"
FREQUENCY = "M"
DATA_PATH = "/Users/pg/Library/CloudStorage/Dropbox-CBS/Pipe Galera/fastest_data"

In [ ]:
authorization = (USERNAME, APIKEY)
headers = {'Accept': 'application/json', 'Content-Type': 'application/json'}

## 2. Define Danish companies (ISINs)

In [ ]:
danish_companies = {
    "Novo Nordisk B": "DK0062498333",
    "DSV": "DK0060079531",
    "Danske Bank": "DK0010274414",
    "Vestas Wind Systems": "DK0061539921",
    "Ørsted": "DK0060094928",
    "Carlsberg B": "DK0010181759",
    "A.P. Møller - Mærsk A": "DK0010244425",
    "A.P. Møller - Mærsk B": "DK0010244508",
    "Genmab": "DK0010272202",
    "Coloplast": "DK0060448595",
    "Tryg": "DK0060636678",
    "Pandora": "DK0060252690",
}

In [ ]:
company_ids = list(danish_companies.values())
company_names = {v: k for k, v in danish_companies.items()}

print(f"Companies to query: {len(danish_companies)}")
for name, isin in danish_companies.items():
    print(f"  {name:30s} {isin}")

## Helper functions

In [ ]:
def reorder_columns(df, leading_cols=("date", "requestCompany")):
    """Move leading_cols to the front of the DataFrame, keeping the rest in original order."""
    front = [c for c in leading_cols if c in df.columns]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]


def fetch_esg(url, request_body, label, date_col="date"):
    """Generic fetch for any FactSet ESG endpoint. Returns a cleaned DataFrame."""
    response = requests.post(
        url=url,
        data=json.dumps(request_body),
        auth=authorization,
        headers=headers,
        verify=False
    )
    print(f"{label} - HTTP Status: {response.status_code}")

    if response.status_code != 200:
        print(f"Error: {response.text[:300]}")
        return pd.DataFrame()

    df = json_normalize(response.json()['data'])
    df['requestCompany'] = df['requestId'].map(company_names)
    df = reorder_columns(df, leading_cols=(date_col, "requestCompany"))
    print(f"Records: {len(df)}, Columns: {len(df.columns)}")
    time.sleep(0.15)
    return df


def fetch_scores(score_type, fields=None):
    """Fetch Truvalue scores for all Danish companies."""
    if fields is None:
        fields = ["TOPLEVEL", "PILLARS", "DIMENSIONS", "SASBCATEGORIES"]

    return fetch_esg(
        url=ESG_SCORES_URL,
        request_body={
            "data": {
                "ids": company_ids,
                "scoreType": score_type,
                "fields": fields,
                "startDate": START_DATE,
                "endDate": END_DATE,
                "frequency": FREQUENCY,
                "calendar": "FIVEDAY"
            }
        },
        label=f"{score_type} scores",
    )


def fetch_spotlights():
    """Fetch Truvalue spotlights for all Danish companies."""
    return fetch_esg(
        url=ESG_SPOTLIGHTS_URL,
        request_body={
            "data": {
                "ids": company_ids,
                "startDate": START_DATE,
                "endDate": END_DATE,
                "categories": ["AllCategories"],
                "primaryOnly": True,
                "isRemoved": False
            }
        },
        label="Spotlights",
        date_col="liveDate",
    )


def fetch_articles():
    """Fetch Truvalue articles for all Danish companies."""
    return fetch_esg(
        url=ESG_ARTICLES_URL,
        request_body={
            "data": {
                "ids": company_ids,
                "categories": ["AllCategories"],
                "startDate": START_DATE,
                "endDate": END_DATE,
                "dateOf": "PUBLICATION"
            }
        },
        label="Articles",
        date_col="datePublication",
    )


def save_file(dataframe, type_name):
    """Save a DataFrame to CSV in the data directory."""
    file_name = f"danish_top_companies_{type_name}_{START_DATE}_{END_DATE}.csv"
    dataframe.to_csv(f"{DATA_PATH}/{file_name}", index=False)
    print(f"Saved: {file_name}")

## 3. Truvalue Scores - PULSE across all SASB categories

Retrieve monthly PULSE scores for 2024 with all 26 SASB categories. The API accepts up to 1000 IDs per request, so all companies fit in a single call.

In [ ]:
pulse_df = fetch_scores("PULSE")
display(pulse_df.head(10))

In [ ]:
save_file(pulse_df, "pulse")

## 4. Truvalue Scores - INSIGHT (long-term ESG track record)

In [ ]:
insight_df = fetch_scores("INSIGHT")
display(insight_df.head(10))

In [ ]:
save_file(insight_df, "insight")

## 5. Truvalue Scores - MOMENTUM (12-month trend)

In [ ]:
momentum_df = fetch_scores("MOMENTUM")
display(momentum_df.head(10))

In [ ]:
save_file(momentum_df, "momentum")

## 6. Truvalue Scores - RANKS (Leader to Laggard)

In [ ]:
ranks_df = fetch_scores("RANKS", fields=["TOPLEVEL"])
display(ranks_df.head(10))

In [ ]:
save_file(ranks_df, "ranks")

## 7. Latest ESG Rank per company

Extract the most recent rank for each company to get a snapshot comparison.

In [ ]:
if 'ranks_df' in dir() and not ranks_df.empty:
    # Get the latest date per company
    latest_ranks = (
        ranks_df
        .sort_values('date')
        .groupby('requestCompany')
        .last()
        .reset_index()
    )
    display_cols = [c for c in ['requestCompany', 'date', 'allCategoriesEsgRank',
                                'allCategoriesIndPctl', 'allCategoriesAdjInsight',
                                'materialityIndPctl', 'materialityAdjInsight'] if c in latest_ranks.columns]
    latest_ranks = latest_ranks[display_cols].sort_values('requestCompany')
    print("Latest ESG Rank per company:\n")
    display(latest_ranks.reset_index(drop=True))
else:
    print("No ranks data available.")

## 8. Truvalue Spotlights - significant ESG events

Retrieve the most significant positive and negative ESG events across all SASB categories for 2024.

In [ ]:
spotlights_df = fetch_spotlights()
print(f"\nSpotlights per company:")
print(spotlights_df['requestCompany'].value_counts().to_string())
display(spotlights_df.head(10))

In [ ]:
save_file(spotlights_df, "spotlights")

## 9. Spotlights summary - count by company and ESG pillar

In [ ]:
if 'spotlights_df' in dir() and not spotlights_df.empty:
    pillar_col = 'spotlightPillar' if 'spotlightPillar' in spotlights_df.columns else None
    cat_col = 'spotlightCategory' if 'spotlightCategory' in spotlights_df.columns else None

    if pillar_col:
        pillar_summary = spotlights_df.groupby(['requestCompany', pillar_col]).size().unstack(fill_value=0)
        pillar_summary['TOTAL'] = pillar_summary.sum(axis=1)
        pillar_summary = pillar_summary.sort_values('TOTAL', ascending=False)
        print("Spotlight counts by company and ESG pillar:\n")
        display(pillar_summary)

    if cat_col:
        print("\nSpotlight counts by SASB category (top 15):\n")
        print(spotlights_df[cat_col].value_counts().head(15).to_string())
else:
    print("No spotlights data available.")

## 10. Truvalue Articles - ESG-related news articles

Retrieve articles across all SASB categories for 2024, sorted by publication date.

In [ ]:
articles_df = fetch_articles()
print(f"\nArticles per company:")
print(articles_df['requestCompany'].value_counts().to_string())
display(articles_df.head(10))

In [ ]:
save_file(articles_df, "articles")

## 11. Articles summary - monthly volume per company

In [ ]:
if 'articles_df' in dir() and not articles_df.empty:
    date_col = 'datePublication' if 'datePublication' in articles_df.columns else None
    if date_col:
        articles_df['month'] = pd.to_datetime(articles_df[date_col]).dt.to_period('M')
        monthly_articles = articles_df.groupby(['requestCompany', 'month']).size().unstack(fill_value=0)
        monthly_articles['TOTAL'] = monthly_articles.sum(axis=1)
        monthly_articles = monthly_articles.sort_values('TOTAL', ascending=False)
        print("Monthly article volume per company:\n")
        display(monthly_articles)
    else:
        print("No datePublication column found. Available columns:", list(articles_df.columns))
else:
    print("No articles data available.")